<a href="https://colab.research.google.com/github/sruthi-kurra/llm-distillation/blob/main/notebooks/02_generate_teacher_labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cell 1 — Setup: load teacher model and dataset

In [ ]:
!pip install transformers datasets evaluate rouge_score sentencepiece -q

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset

print(f"GPU available: {torch.cuda.is_available()}")

# Load teacher
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
teacher = AutoModelForSeq2SeqLM.from_pretrained(model_name)
teacher = teacher.cuda()
teacher.eval()

print(f"Teacher loaded: {model_name}")

# Load dataset
dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")
train_dataset = dataset['train'].select(range(20000))

print(f"Train subset: {len(train_dataset):,} articles")

## Cell 2 — Generate teacher summaries on training subset

In [ ]:
import time
import json
import os

os.makedirs("results", exist_ok=True)

def generate_summary(article, max_input_length=512, max_output_length=128):
    inputs = tokenizer(
        article,
        max_length=max_input_length,
        truncation=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        summary_ids = teacher.generate(
            inputs["input_ids"],
            num_beams=4,
            max_length=max_output_length,
            early_stopping=True
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

N_SAMPLES = 2000
subset = train_dataset.select(range(N_SAMPLES))

print(f"Generating teacher summaries for {N_SAMPLES} articles...")
start_time = time.time()

teacher_summaries = []

for i, example in enumerate(subset):
    summary = generate_summary(example["article"])

    teacher_summaries.append({
        "article": example["article"],
        "reference_summary": example["highlights"],
        "teacher_summary": summary
    })

    if (i + 1) % 100 == 0:
        elapsed = time.time() - start_time
        print(f"Processed {i+1}/{N_SAMPLES} | {elapsed:.1f}s")
        with open("results/teacher_summaries_partial.json", "w") as f:
            json.dump(teacher_summaries, f)

with open("results/teacher_summaries.json", "w") as f:
    json.dump(teacher_summaries, f)

total_time = time.time() - start_time
print(f"\nDone! {len(teacher_summaries)} summaries generated.")
print(f"Average: {total_time/N_SAMPLES:.2f}s/sample")

## Cell 3 — Save teacher summaries to JSON

In [ ]:
import json

# Verify the saved file
with open('results/teacher_summaries.json', 'r') as f:
    saved_summaries = json.load(f)

print(f"Total summaries saved: {len(saved_summaries)}")
print(f"\nExample entry:")
print(f"Article (first 200 chars): {saved_summaries[0]['article'][:200]}")
print(f"\nReference summary: {saved_summaries[0]['reference_summary']}")
print(f"\nTeacher summary: {saved_summaries[0]['teacher_summary']}")

# File size check
import os
size_mb = os.path.getsize('results/teacher_summaries.json') / (1024*1024)
print(f"\nFile size: {size_mb:.1f} MB")

In [ ]:
from google.colab import files
files.download('results/teacher_summaries.json')